# Q4 — Naive Bayes

In [1]:
import sys
import sklearn
print("Python:", sys.executable)
print("scikit-learn:", sklearn.__version__)

Python: /usr/bin/python3
scikit-learn: 1.4.1.post1


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
from html import escape
import numpy as np
from IPython.display import HTML, display

# Support launching from the notebook directory or repository root.
candidates = [Path('../Q4-train.csv'), Path('Q4-train.csv'), Path('Homework/Homework1/Q4-train.csv')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError('Run from Homework1-Solutions or the repository root.')

def table(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

with data_path.open(newline='') as f:
    raw_rows = list(csv.reader(f))
widths = Counter(len(row) for row in raw_rows)
print('Source:', data_path)
print('Rows:', len(raw_rows))
print('Column counts (width: number of rows):', dict(widths))
assert raw_rows and set(widths) == {25}, 'Expected 25 columns in every row; inspect malformed rows before continuing.'
names = [f'F{i}' for i in range(1, 25)] + ['Class']
table(names, raw_rows[:5])

Source: ../Q4-train.csv
Rows: 700
Column counts (width: number of rows): {25: 700}


F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,Class
1,6,4,12,5,5,3,4,1,67,3,2,1,2,1,0,0,1,0,0,1,0,0,1,1
2,48,2,60,1,3,2,2,1,22,3,1,1,1,1,0,0,1,0,0,1,0,0,1,2
4,12,4,21,1,4,3,3,1,49,3,1,2,1,1,0,0,1,0,0,1,0,1,0,1
1,42,2,79,1,4,3,4,2,45,3,1,2,1,1,0,0,0,0,0,0,0,0,1,1
1,24,3,49,1,3,3,4,4,53,3,2,2,1,1,1,0,1,0,0,0,0,0,1,2


In [3]:
missing_tokens = {'', '?', 'na', 'n/a', 'nan', 'null', 'none', 'missing'}
data = np.full((len(raw_rows), 25), np.nan)
missing = np.zeros(25, dtype=int)
invalid = np.zeros(25, dtype=int)
infinite = np.zeros(25, dtype=int)
issues = []
for i, row in enumerate(raw_rows):
    for j, token in enumerate(row):
        token = token.strip()
        if token.lower() in missing_tokens:
            missing[j] += 1
            issues.append((i + 1, names[j], repr(token), 'missing token'))
            continue
        try:
            value = float(token)
        except ValueError:
            invalid[j] += 1
            issues.append((i + 1, names[j], repr(token), 'nonnumeric'))
            continue
        if not np.isfinite(value):
            infinite[j] += 1
            issues.append((i + 1, names[j], repr(token), 'nonfinite'))
        else:
            data[i, j] = value

table(['Column', 'Missing tokens', 'Nonnumeric', 'Nonfinite', 'Usable values'],
      [(name, missing[j], invalid[j], infinite[j], np.isfinite(data[:, j]).sum())
       for j, name in enumerate(names)])
print('Total flagged cells:', len(issues))
if issues:
    table(['CSV row (1-based)', 'Column', 'Token', 'Issue'], issues[:20])
X, y = data[:, :-1], data[:, -1]
print('Feature shape:', X.shape, '| Label shape:', y.shape)

Column,Missing tokens,Nonnumeric,Nonfinite,Usable values
F1,0,0,0,700
F2,0,0,0,700
F3,0,0,0,700
F4,0,0,0,700
F5,0,0,0,700
F6,0,0,0,700
F7,0,0,0,700
F8,0,0,0,700
F9,0,0,0,700
F10,0,0,0,700


Total flagged cells: 0
Feature shape: (700, 24) | Label shape: (700,)


In [4]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

train = np.loadtxt(data_path, delimiter=',')
test = np.loadtxt(data_path.with_name('Q4-test.csv'), delimiter=',')
print('Training rows and columns:', train.shape)
print('Test rows and columns:', test.shape)
assert train.shape[1] == test.shape[1] == 25

X_train, y_train = train[:, :-1], train[:, -1]
X_test, y_test = test[:, :-1], test[:, -1]

model = GaussianNB()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f'Accuracy:        {accuracy_score(y_test, predictions):.3f}')
print(f'Macro-precision: {precision_score(y_test, predictions, average="macro", zero_division=0):.3f}')
print(f'Macro-recall:    {recall_score(y_test, predictions, average="macro", zero_division=0):.3f}')
print(f'Macro-F1:        {f1_score(y_test, predictions, average="macro", zero_division=0):.3f}')
print('Confusion matrix: rows = actual [1, 2], columns = predicted [1, 2]')
print(confusion_matrix(y_test, predictions, labels=[1, 2]))

wrong_rows = np.flatnonzero(predictions != y_test)
for i in wrong_rows[:5]:
    print(f'\nTest row {i + 1}: actual={y_test[i]:g}, predicted={predictions[i]:g}')
    print('F1 through F24:', X_test[i].astype(int))

Training rows and columns: (700, 25)
Test rows and columns: (300, 25)
Accuracy:        0.740
Macro-precision: 0.701
Macro-recall:    0.714
Macro-F1:        0.706
Confusion matrix: rows = actual [1, 2], columns = predicted [1, 2]
[[162  45]
 [ 33  60]]

Test row 1: actual=2, predicted=1
F1 through F24: [ 4 12  2 11  3  3  2  4  3 29  3  1  1  1  1  0  0  1  0  1  0  0  1  0]

Test row 2: actual=2, predicted=1
F1 through F24: [ 1 48  4 63  1  5  3  4  4 46  3  2  1  2  1  0  1  1  0  0  0  0  0  1]

Test row 5: actual=1, predicted=2
F1 through F24: [ 2 27  2 25  1  2  2  1  2 32  3  1  2  2  1  0  0  1  0  0  1  0  0  1]

Test row 21: actual=2, predicted=1
F1 through F24: [ 3  9  0 13  1  2  3  2  3 34  3  2  1  2  1  0  0  1  0  0  1  0  0  0]

Test row 31: actual=1, predicted=2
F1 through F24: [ 2 24  3 64  1  2  3  2  3 33  3  1  1  1  1  0  0  1  0  0  1  0  0  1]


In [5]:
summary = []
for j, name in enumerate(names[:-1]):
    values = X[np.isfinite(X[:, j]), j]
    unique = np.unique(values)
    if not len(values):
        summary.append([name, 0, len(X), 0, 'no usable values'] + ['—'] * 7)
        continue
    observed = ', '.join(f'{v:g}' for v in unique) if len(unique) <= 20 else f'{unique[0]:g} … {unique[-1]:g} (range)'
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    stats = [values.min(), values.max(), values.mean(), values.std(ddof=1) if len(values) > 1 else np.nan, q1, median, q3]
    summary.append([name, len(values), len(X) - len(values), len(unique), observed] + [f'{v:.3f}' for v in stats])
table(['Feature', 'Valid', 'Unavailable', 'Unique', 'Observed values / range',
       'Min', 'Max', 'Mean', 'Sample SD', 'Q1', 'Median', 'Q3'], summary)

Feature,Valid,Unavailable,Unique,Observed values / range,Min,Max,Mean,Sample SD,Q1,Median,Q3
F1,700,0,4,"1, 2, 3, 4",1.000,4.000,2.586,1.244,1.000,2.000,4.000
F2,700,0,32,4 … 72 (range),4.000,72.000,20.653,12.280,12.000,18.000,24.000
F3,700,0,5,"0, 1, 2, 3, 4",0.000,4.000,2.543,1.071,2.000,2.000,4.000
F4,700,0,112,3 … 159 (range),3.000,159.000,31.823,27.279,13.750,23.000,39.000
F5,700,0,5,"1, 2, 3, 4, 5",1.000,5.000,2.061,1.551,1.000,1.000,3.000
F6,700,0,5,"1, 2, 3, 4, 5",1.000,5.000,3.366,1.195,3.000,3.000,4.000
F7,700,0,4,"1, 2, 3, 4",1.000,4.000,2.686,0.704,2.000,3.000,3.000
F8,700,0,4,"1, 2, 3, 4",1.000,4.000,2.809,1.115,2.000,3.000,4.000
F9,700,0,4,"1, 2, 3, 4",1.000,4.000,2.367,1.055,1.000,2.000,3.000
F10,700,0,52,19 … 75 (range),19.000,75.000,35.323,11.320,27.000,33.000,41.000


In [6]:
for j, name in enumerate(names[:-1]):
    values, counts = np.unique(X[np.isfinite(X[:, j]), j], return_counts=True)
    if len(values) <= 20:
        print(f"{name}: " + ', '.join(f'{v:g}: {n}' for v, n in zip(values, counts)))

labels, counts = np.unique(y[np.isfinite(y)], return_counts=True)
table(['Class code', 'Count', 'Percent of all training rows'],
      [(f'{label:g}', count, f'{100 * count / len(y):.3f}%') for label, count in zip(labels, counts)])
constant = [names[j] for j in range(24) if len(np.unique(X[np.isfinite(X[:, j]), j])) == 1]
binary = [names[j] for j in range(24) if set(X[np.isfinite(X[:, j]), j]) == {0, 1}]
print('Constant features:', constant)
print('Features with exactly {0, 1}:', binary)
print('All usable feature values are integer-valued:', bool(np.all(X[np.isfinite(X)] == np.floor(X[np.isfinite(X)]))))

complete = data[np.isfinite(data).all(axis=1)]
print('Complete rows used for duplicate checks:', len(complete))
print('Repeated complete rows beyond first occurrence:', len(complete) - len(np.unique(complete, axis=0)))
print('Repeated feature vectors beyond first occurrence:', len(complete) - len(np.unique(complete[:, :-1], axis=0)))
labels_by_features = defaultdict(set)
for row in complete:
    labels_by_features[tuple(row[:-1])].add(row[-1])
print('Feature vectors with conflicting labels:', sum(len(labels) > 1 for labels in labels_by_features.values()))

F1: 1: 183, 2: 197, 3: 47, 4: 273
F3: 0: 28, 1: 30, 2: 376, 3: 66, 4: 200
F5: 1: 427, 2: 77, 3: 42, 4: 34, 5: 120
F6: 1: 44, 2: 118, 3: 244, 4: 126, 5: 168
F7: 1: 34, 2: 216, 3: 386, 4: 64
F8: 1: 100, 2: 214, 3: 106, 4: 280
F9: 1: 199, 2: 154, 3: 238, 4: 109
F11: 1: 98, 2: 32, 3: 570
F12: 1: 452, 2: 225, 3: 19, 4: 4
F13: 1: 596, 2: 104
F14: 1: 422, 2: 278
F15: 1: 674, 2: 26
F16: 0: 543, 1: 157
F17: 0: 635, 1: 65
F18: 0: 67, 1: 633
F19: 0: 670, 1: 30
F20: 0: 581, 1: 119
F21: 0: 197, 1: 503
F22: 0: 687, 1: 13
F23: 0: 563, 1: 137
F24: 0: 255, 1: 445


Class code,Count,Percent of all training rows
1,493,70.429%
2,207,29.571%


Constant features: []
Features with exactly {0, 1}: ['F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F24']
All usable feature values are integer-valued: True
Complete rows used for duplicate checks: 700
Repeated complete rows beyond first occurrence: 0
Repeated feature vectors beyond first occurrence: 0
Feature vectors with conflicting labels: 0
